In [6]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU Memory Growth Enabled")
  except RuntimeError as e:
    print(e)

GPU Memory Growth Enabled


In [7]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense , Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import GlobalAveragePooling2D


In [8]:
train_ds=keras.utils.image_dataset_from_directory(
    directory="/home/conqueror/college dl/dogsvscats/train",
    labels="inferred",label_mode="int",
    batch_size=128,image_size=(128,128),
    color_mode="rgb"
)
test_ds=keras.utils.image_dataset_from_directory(
    directory="/home/conqueror/college dl/dogsvscats/test",
    labels="inferred",label_mode="int",
    batch_size=128,image_size=(128,128),
    color_mode="rgb"
)
data_argumentaion=tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

dataset_size=len(train_ds)
train_size=int(0.8*dataset_size)
AUTOTUNE = tf.data.AUTOTUNE
train=train_ds.take(train_size).cache().prefetch(buffer_size=AUTOTUNE)
val=train_ds.skip(train_size).cache().prefetch(buffer_size=AUTOTUNE)


Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.


In [9]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Lambda
from tensorflow.keras.models import Sequential

# 1. Base Model ko Load karein (VGG16 ki jagah MobileNetV2)
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(128, 128, 3)
)
# Feature Extraction ke liye model freeze karein (Process bilkul same hai!)
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable=False
    

# 2. Apna naya Model banayein
model = Sequential()
model.add(data_argumentaion) # Aapka augmentation
model.add(Lambda(preprocess_input)) # MobileNetV2 ka apna preprocess function
model.add(base_model)

# Flatten() ki jagah GlobalAveragePooling2D! (Overfitting rokne ke liye)
model.add(GlobalAveragePooling2D())
model.add(Dense(256, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer="Adam", loss='binary_crossentropy', metrics=["accuracy"])

# Kam se kam 15 epochs chalayein achi accuracy ke liye
history = model.fit(train, epochs=15, validation_data=val)


Epoch 1/15
125/125 [==============================] - 18s 104ms/step - loss: 0.1400 - accuracy: 0.9440 - val_loss: 0.0784 - val_accuracy: 0.9745
Epoch 2/15
125/125 [==============================] - 11s 86ms/step - loss: 0.0869 - accuracy: 0.9629 - val_loss: 0.0626 - val_accuracy: 0.9780
Epoch 3/15
125/125 [==============================] - 11s 87ms/step - loss: 0.0733 - accuracy: 0.9721 - val_loss: 0.0619 - val_accuracy: 0.9793
Epoch 4/15
125/125 [==============================] - 11s 87ms/step - loss: 0.0639 - accuracy: 0.9748 - val_loss: 0.0617 - val_accuracy: 0.9800
Epoch 5/15
125/125 [==============================] - 11s 88ms/step - loss: 0.0574 - accuracy: 0.9783 - val_loss: 0.0528 - val_accuracy: 0.9812
Epoch 6/15
125/125 [==============================] - 11s 87ms/step - loss: 0.0540 - accuracy: 0.9809 - val_loss: 0.0520 - val_accuracy: 0.9820
Epoch 7/15
125/125 [==============================] - 11s 87ms/step - loss: 0.0528 - accuracy: 0.9806 - val_loss: 0.0580 - val_accuracy

In [10]:
model.evaluate(test_ds)

40/40 [==============================] - 3s 66ms/step - loss: 0.0900 - accuracy: 0.9794


[0.08998032659292221, 0.9793999791145325]